# SobraBoa — IA de Triagem de Alimentos

**Projeto:** Global Solution 2026/2 — CitizenTech  
**Disciplina:** Disruptive Architectures: IoT, IoB & Generative IA  
**Grupo:** Felipe Kirschner Modesto (RM561810), João Victor L. O. Resende (RM565139), Pedro Vaz Ferreira (RM566551), Vitor Dias dos Santos (RM565422)

## O que essa IA faz

O SobraBoa é uma plataforma de redistribuição de excedentes alimentares. Quando um doador (padaria, feira, restaurante) vai publicar uma doação, ele tira foto do alimento. Essa IA:

1. **Analisa a imagem** (Visão Computacional) e classifica o alimento
2. **Retorna um JSON estruturado** com categoria, validade estimada, necessidade de refrigeração e se está próprio para doação
3. **Abre um chat contextual** (IA Generativa) onde o doador pode tirar dúvidas sobre aquele alimento específico

O contexto fixo de categorias e regras do sistema funciona como um **RAG básico** — a IA sempre responde dentro das regras da plataforma.

## Arquitetura

```
Foto do alimento
      │
      ▼
Gemini 3.5 Flash-Lite  ◄─── System Prompt (categorias + regras SobraBoa = RAG básico)
      │
      ├──► JSON estruturado (categoria, validade, refrigeração, aprovação)
      │
      └──► Chat conversacional com contexto do alimento analisado
```

In [ ]:
# Célula 1 — Instalação das dependências
!pip install -q google-generativeai requests Pillow

In [ ]:
# Célula 2 — Imports e configuração do modelo
import google.generativeai as genai
import requests
from PIL import Image
from io import BytesIO
import json
from google.colab import userdata

# Adiciona a chave no Secrets do Colab com o nome GEMINI_API_KEY
# (ícone de chave no menu lateral esquerdo)
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

modelo = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    system_instruction="""
Você é o assistente de triagem de alimentos do SobraBoa, uma plataforma brasileira
de redistribuição de excedentes alimentares que conecta pequenos comércios
(padarias, feiras, restaurantes) a ONGs e cozinhas comunitárias.

Seu papel é analisar alimentos que doadores desejam publicar na plataforma e retornar
uma avaliação estruturada com base nas seguintes categorias cadastradas no sistema:
- Pães e Confeitaria
- Hortifrúti (frutas, legumes, verduras)
- Cozinhados e Refeições Prontas
- Laticínios
- Carnes e Proteínas
- Bebidas
- Mercearia (enlatados, grãos, massas)
- Outros

Regras de negócio que você deve aplicar:
1. Alimentos visivelmente estragados (mofo, deterioração, coloração anormal) NÃO são
   próprios para doação e devem ser recusados com explicação clara.
2. Alimentos que requerem refrigeração (laticínios, carnes, cozinhados com proteína)
   devem ser marcados com alerta de cadeia fria.
3. A validade estimada deve ser conservadora — prefira subestimar a superestimar.
4. Oriente o doador sobre embalagem e transporte quando necessário.
5. Base legal: Lei nº 14.016/2020 (Brasil) permite doação de alimentos excedentes
   por estabelecimentos comerciais desde que próprios para consumo.

Sempre responda em português brasileiro, com linguagem simples e direta.
"""
)

print("✅ Modelo configurado com sucesso!")

In [ ]:
# Célula 3 — Funções auxiliares

def carregar_imagem_url(url):
    """Baixa uma imagem da internet e converte para o formato PIL."""
    resposta = requests.get(url, timeout=10)
    imagem = Image.open(BytesIO(resposta.content))
    return imagem


def analisar_alimento(imagem, descricao_extra=""):
    """
    Analisa a imagem do alimento e retorna JSON estruturado.
    descricao_extra: informação adicional que o doador pode fornecer.
    """
    prompt = f"""
Analise esta imagem de alimento que um doador deseja publicar na plataforma SobraBoa.

{f'Informação adicional do doador: {descricao_extra}' if descricao_extra else ''}

Retorne SOMENTE um JSON válido com esta estrutura (sem markdown, sem texto extra):
{{
  "proprio_para_doacao": true ou false,
  "categoria": "nome da categoria do sistema",
  "requer_refrigeracao": true ou false,
  "validade_estimada": "descrição da validade (ex: consumir em até 1 dia)",
  "cuidados_transporte": "orientações de embalagem e transporte",
  "observacoes": "observações importantes para o receptor",
  "motivo_recusa": "preencher apenas se proprio_para_doacao for false"
}}
"""
    resposta = modelo.generate_content([prompt, imagem])

    # Limpa a resposta caso venha com bloco markdown
    texto = resposta.text.strip()
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]

    resultado = json.loads(texto.strip())
    return resultado


def exibir_resultado(resultado, titulo=""):
    """Exibe o resultado da análise de forma legível."""
    print(f"\n{'='*55}")
    if titulo:
        print(f"🔍 TESTE: {titulo}")
    print(f"{'='*55}")

    status = "✅ APROVADO PARA DOAÇÃO" if resultado["proprio_para_doacao"] else "❌ RECUSADO"
    print(f"Status:              {status}")
    print(f"Categoria:           {resultado['categoria']}")
    print(f"Refrigeração:        {'⚠️  NECESSÁRIA' if resultado['requer_refrigeracao'] else 'Não necessária'}")
    print(f"Validade estimada:   {resultado['validade_estimada']}")
    print(f"Transporte:          {resultado['cuidados_transporte']}")
    print(f"Observações:         {resultado['observacoes']}")

    if not resultado["proprio_para_doacao"] and resultado.get("motivo_recusa"):
        print(f"Motivo da recusa:    {resultado['motivo_recusa']}")

    print(f"{'='*55}\n")
    return resultado


print("✅ Funções carregadas!")

## Testes de Visão Computacional

Três cenários para validar a IA:
- **Teste 1:** Alimento bom → deve ser aprovado
- **Teste 2:** Alimento estragado → deve ser recusado
- **Teste 3:** Alimento com cadeia fria → deve ser aprovado com alerta de refrigeração

In [ ]:
# Célula 4 — Teste 1: Alimento bom (pão fresco)
url_pao = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/33/French_bread_DSC09293.jpg/1280px-French_bread_DSC09293.jpg"

print("Carregando imagem...")
imagem_pao = carregar_imagem_url(url_pao)
print("Analisando com Gemini...")

resultado_pao = analisar_alimento(imagem_pao, descricao_extra="Saiu do forno há 2 horas")
exibir_resultado(resultado_pao, "Pão francês fresco")

In [ ]:
# Célula 5 — Teste 2: Alimento impróprio (mofado)
url_mofado = "https://upload.wikimedia.org/wikipedia/commons/thumb/f/fc/Mouldy_bread.jpg/1280px-Mouldy_bread.jpg"

print("Carregando imagem...")
imagem_mofada = carregar_imagem_url(url_mofado)
print("Analisando com Gemini...")

resultado_mofado = analisar_alimento(imagem_mofada)
exibir_resultado(resultado_mofado, "Alimento com mofo visível")

In [ ]:
# Célula 6 — Teste 3: Alimento com cadeia fria (laticínio)
url_queijo = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Cheese_platter.jpg/1280px-Cheese_platter.jpg"

print("Carregando imagem...")
imagem_queijo = carregar_imagem_url(url_queijo)
print("Analisando com Gemini...")

resultado_queijo = analisar_alimento(imagem_queijo, descricao_extra="Queijos artesanais, foram refrigerados até agora")
exibir_resultado(resultado_queijo, "Laticínio — alerta de cadeia fria")

## Chat Conversacional (IA Generativa)

Após a análise da imagem, o doador pode tirar dúvidas sobre aquele alimento específico. O chat usa o JSON da análise como contexto — é aqui que entra a IA generativa conversacional integrada ao fluxo da solução.

In [ ]:
# Célula 7 — Chat contextual com base no alimento analisado
# Usa o resultado do Teste 1 (pão fresco) como contexto

contexto_alimento = json.dumps(resultado_pao, ensure_ascii=False, indent=2)
historico_chat = []


def chat_assistente(mensagem_usuario):
    """Chat conversacional com contexto do alimento já analisado."""
    historico_texto = ""
    for msg in historico_chat:
        prefixo = "DOADOR" if msg["role"] == "user" else "ASSISTENTE"
        historico_texto += f"\n{prefixo}: {msg['texto']}"

    prompt = f"""
O doador já analisou o seguinte alimento na plataforma SobraBoa:
{contexto_alimento}

Histórico da conversa:{historico_texto if historico_texto else ' (início da conversa)'}

DOADOR: {mensagem_usuario}

Responda de forma direta e prática. Máximo 3 linhas.
"""

    resposta = modelo.generate_content(prompt)
    texto_resposta = resposta.text.strip()

    historico_chat.append({"role": "user", "texto": mensagem_usuario})
    historico_chat.append({"role": "model", "texto": texto_resposta})

    return texto_resposta


print("Chat iniciado com contexto do pão fresco (Teste 1)")
print("=" * 55)

perguntas = [
    "Posso embalar em saco plástico comum?",
    "O receptor pode buscar em até 3 horas?",
    "Preciso colocar alguma etiqueta de identificação?"
]

for pergunta in perguntas:
    print(f"\n👤 Doador: {pergunta}")
    resposta = chat_assistente(pergunta)
    print(f"🤖 Assistente: {resposta}")

In [ ]:
# Célula 8 — Resumo dos resultados (evidência de avaliação)
print("\n" + "="*55)
print("  RESUMO DOS TESTES — SobraBoa IA")
print("="*55)

testes = [
    ("Pão francês fresco",        resultado_pao),
    ("Alimento com mofo",         resultado_mofado),
    ("Queijo / laticínio",        resultado_queijo),
]

for nome, resultado in testes:
    status = "✅ Aprovado" if resultado["proprio_para_doacao"] else "❌ Recusado"
    refrig = "⚠️ Sim" if resultado["requer_refrigeracao"] else "Não"
    print(f"\n{nome}")
    print(f"  Status:    {status}")
    print(f"  Categoria: {resultado['categoria']}")
    print(f"  Refrig.:   {refrig}")
    print(f"  Validade:  {resultado['validade_estimada']}")

print("\n" + "="*55)
print("  Técnicas aplicadas:")
print("  ✓ Visão Computacional (análise de imagem)")
print("  ✓ IA Generativa (chat contextual)")
print("  ✓ RAG básico (system prompt com regras do domínio)")
print(f"  Modelo: gemini-3.5-flash-lite")
print("="*55)